<a href="https://colab.research.google.com/github/Mahsa-Goudarzi/vap-time-indexed/blob/main/vap_notebooks/base_no_time_indexed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pulp
import numpy as np

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DATA
# ═══════════════════════════════════════════════════════════════════════════

# carriers and trips
carriers = [0, 1]          # two carrier companies: C1, C2
trips    = [0, 1, 2, 3, 4] # all the trips

# trips of each carrier
trips_of_carrier = {
    0: [0, 1],   # C1
    1: [2, 3, 4],   # C2
}

# vehicles of each carrier: (carrier -> list of vehicle ids)
vehicles_of_carrier = {
    0: [0, 1, 2],   # C1: vehicles 0,1,2
    1: [3, 4, 5],   # C2: vehicles 3,4,5
}

#if vehicle v is electric: is_electric[v] = True
is_electric = {0: False, 1: False, 2: True,
               3: True,  4: True,  5: False}

all_vehicles = [v for vlist in vehicles_of_carrier.values() for v in vlist]

# ── scalar parameters ──────────────────────────────────────────────────────
alpha  = 1.0   # revenue scaling coefficient
beta   = 0.8   # emission cost attribution coefficient
mu     = 80.0  # average truck speed  (km/h)
c_E    = 0.2   # cost per gram CO2    (€/g)

# unit revenue I^r of each carrier r (€/km)
I = {0: 1.8, 1: 1.8}

# trip distances  d_n  (km)
d = {0: 150, 1: 100, 2: 80, 3: 120, 4: 130}

# unit operating cost  c^{rv}_O  (€/km) — c[carrier][vehicle]
# for vehicles NOT owned by carrier r → large penalty (1000) prevents use
op_cost_all_diesels = 0.8
op_cost_all_electrics = 0.7

op_cost = {
    0: {0: op_cost_all_diesels, 1: op_cost_all_diesels, 2: op_cost_all_electrics},   # carrier 0 owns V0,V1,V2
    1: {3: op_cost_all_electrics, 4: op_cost_all_electrics, 5: op_cost_all_diesels},   # carrier 1 owns V3,V4,V5
}
op_cost_full = {
    (0, 0): op_cost_all_diesels, (0, 1): op_cost_all_diesels, (0, 2): op_cost_all_electrics,
    (0, 3): 1000, (0, 4): 1000, (0, 5): 1000,
    (1, 0): 1000, (1, 1): 1000, (1, 2): 1000,
    (1, 3): op_cost_all_electrics,  (1, 4): op_cost_all_electrics,  (1, 5): op_cost_all_diesels,
}


# ── available time parameters ─────────────────────────
# available time T^v  (hours)
# diesel  → T^v = T^v_drive
# electric → T^v = min(T^v_drive, T^v_charge)

T_drive = {v: 8.0 for v in all_vehicles}
T_charge = {v: 8.0 for v in all_vehicles if is_electric[v]}
T_avail = {}
for v in all_vehicles:
    if is_electric[v]:
        T_avail[v] = min(T_drive[v], T_charge[v])
    else:
        T_avail[v] = T_drive[v]

print("T_avail per vehicle:")
for v in all_vehicles:
    vtype = "Electric" if is_electric[v] else "Diesel"
    print(f"  Vehicle {v} ({vtype}): T_avail = {T_avail[v]} h")

# vehicle capacity  Cap^v  (kg)
Cap = {v: 40.0 for v in all_vehicles}

# weight of goods in trip n p_n  (kg)
p = {n: 25.0 for n in trips}

# emission parameters  (g/km)
E_empty = {0: 0.3, 1: 0.2, 2: 0.3, 3: 0.3, 4: 0.3, 5: 0.3}
E_full  = {0: 0.3, 1: 0.4, 2: 0.3, 3: 0.3, 4: 0.4, 5: 0.3}

# d^v_{o_n}: depot of vehicle v → origin of trip n   (km)   key = (n, v)
#d_origin = {(n, v): 5.0 for n in trips for v in all_vehicles}
d_origin = {
    (0,0):0,   (0,1):0, (0,2):0,  (0,3):50, (0,4):50, (0,5):50,
    (1,0):0, (1,1):0,   (1,2):0,   (1,3):50,  (1,4):50, (1,5):50,
    (2,0):100,  (2,1):100,  (2,2):100,   (2,3):80,   (2,4):80, (2,5):80,
    (3,0):50, (3,1):50, (3,2):50, (3,3):0,   (3,4):0, (3,5):0,
    (4,0):50, (4,1):50, (4,2):50, (4,3):0, (4,4):0, (4,5):0,
}

# d^v_{w_k}: destination of trip k → depot of vehicle v  (km)  key = (k, v)
#d_dest = {(n, v): 5.0 for n in trips for v in all_vehicles}
d_dest = {
    (0,0):150,   (0,1):150, (0,2):150,  (0,3):200, (0,4):200, (0,5):200,
    (1,0):100, (1,1):100,   (1,2):100,   (1,3):150,  (1,4):150, (1,5):150,
    (2,0):50,  (2,1):50,  (2,2):50,   (2,3):0,   (2,4):0, (2,5):0,
    (3,0):170, (3,1):170, (3,2):170, (3,3):120,   (3,4):120, (3,5):120,
    (4,0):180, (4,1):180, (4,2):180, (4,3):130, (4,4):130, (4,5):130,
}

# epsilon_{nk}: repositioning distance dest(n) → origin(k)  (km)
eps = {
    (0,0):0,   (0,1):150, (0,2):60,  (0,3):140, (0,4):140,
    (1,0):150, (1,1):0,   (1,2):0,   (1,3):80,  (1,4):80,
    (2,0):50,  (2,1):50,  (2,2):0,   (2,3):0,   (2,4):0,
    (3,0):170, (3,1):200, (3,2):200, (3,3):0,   (3,4):120,
    (4,0):180, (4,1):180, (4,2):140, (4,3):130, (4,4):0,
}

# O_{nk}: 1 if trips n and k may be combined
#O = {(n, k): 1 for n in trips for k in trips if n != k}
O = {
    (0,0):0, (0,1):0, (0,2):1, (0,3):0, (0,4):0,
    (1,0):0, (1,1):0, (1,2):1, (1,3):0, (1,4):0,
    (2,0):1, (2,1):1, (2,2):0, (2,3):1, (2,4):1,
    (3,0):0, (3,1):0, (3,2):1, (3,3):0, (3,4):0,
    (4,0):0, (4,1):0, (4,2):1, (4,3):0, (4,4):0,
}

# delta_n: compensation cost for trip n  (€)
delta = {0: 20, 1: 20, 2: 10, 3: 10, 4: 10}

# trips ownership
# z^r_n: 1 if trip n belongs to carrier r
z = {}
for r in carriers:
    for n in trips:
        z[(r, n)] = 1 if n in trips_of_carrier[r] else 0

T_avail per vehicle:
  Vehicle 0 (Diesel): T_avail = 8.0 h
  Vehicle 1 (Diesel): T_avail = 8.0 h
  Vehicle 2 (Electric): T_avail = 8.0 h
  Vehicle 3 (Electric): T_avail = 8.0 h
  Vehicle 4 (Electric): T_avail = 8.0 h
  Vehicle 5 (Diesel): T_avail = 8.0 h


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# AUXILIARY FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════

def TOA_single(v, n):
    return d_origin[(n, v)] + d[n] + d_dest[(n, v)]

def TOA_combined(v, n, k):
    return d_origin[(n, v)] + d[n] + eps[(n, k)] + d[k] + d_dest[(k, v)]

def emission_single(v, n):
    if is_electric[v]:
        return 0.0
    return (TOA_single(v, n) * E_empty[v]
            + (E_full[v] - E_empty[v]) * d[n] * p[n] / Cap[v])

def emission_combined(v, n, k):
    if is_electric[v]:
        return 0.0
    return (TOA_combined(v, n, k) * E_empty[v]
            + (E_full[v] - E_empty[v]) * (d[n]*p[n] + d[k]*p[k]) / Cap[v])


In [ ]:

# ═══════════════════════════════════════════════════════════════════════════
# PROBLEM 1 — Initial profit per carrier
# ═══════════════════════════════════════════════════════════════════════════

S0 = {}

for r in carriers:
    prob1 = pulp.LpProblem(f"Problem1_Carrier{r}", pulp.LpMaximize)

    my_trips = trips_of_carrier[r]
    my_vehs  = vehicles_of_carrier[r]

    # Decision variable
    chi = pulp.LpVariable.dicts(
        "chi",
        [(n, v) for n in my_trips for v in my_vehs],
        cat="Binary"
    )

    # Objective
    profit_terms = []
    for n in my_trips:
        revenue   = alpha * d[n] * I[r]

        cost = pulp.lpSum([chi[(n, v)] * (TOA_single(v, n) * op_cost[r][v] + (1 - int(is_electric[v])) * c_E * emission_single(v, n))
        for v in my_vehs])

        profit_terms.append(revenue - cost)

    prob1 += pulp.lpSum(profit_terms)

    # P1-C1: each trip executed exactly once
    for n in my_trips:
        prob1 += pulp.lpSum(chi[(n, v)] for v in my_vehs) == 1

    # P1-C2: trips assigned to carrier r <= fleet size of carrier r
    prob1 += pulp.lpSum(chi[(n, v)]
                        for n in my_trips for v in my_vehs) <= len(my_vehs)

    # P1-C3: each vehicle executes at most one trip
    for v in my_vehs:
        prob1 += pulp.lpSum(chi[(n, v)] for n in my_trips) <= 1

    # P1-C4: battery range for electric vehicles
    for n in my_trips:
        for v in my_vehs:
            if is_electric[v]:
                prob1 += chi[(n, v)] * TOA_single(v, n) <= T_avail[v] * mu

    prob1.solve(pulp.PULP_CBC_CMD(msg=0))

    S0[r] = pulp.value(prob1.objective)
    print(f"Carrier {r}: S0 = {S0[r]:.2f} €  (status: {pulp.LpStatus[prob1.status]})")

Carrier 0: S0 = 69.50 €  (status: Optimal)
Carrier 1: S0 = 106.40 €  (status: Optimal)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# PROBLEM 2 — Cooperative optimisation  (with time indexing)
# ═══════════════════════════════════════════════════════════════════════════

prob2 = pulp.LpProblem("Problem2_Cooperative", pulp.LpMaximize)

# ── Decision variables ──────────────────────────────────
# x[r,v,n]: single trip n by carrier r, vehicle v
x = pulp.LpVariable.dicts(
    "x",
    [(r, v, n)
     for r in carriers
     for v in vehicles_of_carrier[r]
     for n in trips],
    cat="Binary"
)

# y[r,v,n,k]: combined trip (n,k) by carrier r, vehicle v
y = pulp.LpVariable.dicts(
    "y",
    [(r, v, n, k)
     for r in carriers
     for v in vehicles_of_carrier[r]
     for n in trips
     for k in trips if k != n],
    cat="Binary"
)

# ── Profic calculation ────────────────────────────────

def S1(r):
    """profit from single trips"""
    terms = []
    for n in trips:
        for v in vehicles_of_carrier[r]:
            rev  = alpha * d[n] * I[r]
            op_c = TOA_single(v, n) * op_cost_full[(r, v)]
            em_c = beta * c_E * emission_single(v, n) if not is_electric[v] else 0.0
            terms.append((rev - op_c - em_c) * x[(r, v, n)])
    return pulp.lpSum(terms)

def S2(r):
    """profit from combined trips"""
    terms = []
    for n in trips:
        for k in trips:
            if k == n:
                continue
            for v in vehicles_of_carrier[r]:
                rev  = alpha * (d[n] + d[k]) * I[r]
                op_c = TOA_combined(v, n, k) * op_cost_full[(r, v)]
                em_c = beta * c_E * emission_combined(v, n, k) if not is_electric[v] else 0.0
                terms.append((rev - op_c - em_c) * y[(r, v, n, k)])
    return pulp.lpSum(terms)

def S3(r):
    """compensation mechanism"""
    terms = []
    for n in trips:
        for rp in carriers:
            if rp == r:
                continue
            for v in vehicles_of_carrier[rp]:
                # r' executes trip n (owned by r) → r earns delta_n
                terms.append(delta[n] * z[(r, n)]  * x[(rp, v, n)])
            for v in vehicles_of_carrier[r]:
                # r executes trip n (owned by r') → r pays delta_n
                terms.append(-delta[n] * z[(rp, n)] * x[(r, v, n)])

    for n in trips:
        for k in trips:
            if k == n:
                continue
            for rp in carriers:
                if rp == r:
                    continue
                for v in vehicles_of_carrier[rp]:
                    terms.append(delta[n] * z[(r, n)]  * y[(rp, v, n, k)])
                    terms.append(delta[k] * z[(r, k)]  * y[(rp, v, n, k)])
                for v in vehicles_of_carrier[r]:
                    terms.append(-delta[n] * z[(rp, n)] * y[(r, v, n, k)])
                    terms.append(-delta[k] * z[(rp, k)] * y[(r, v, n, k)])
    return pulp.lpSum(terms)

# ── Objective ─────────────────────────────────────────
prob2 += pulp.lpSum(S1(r) + S2(r) + S3(r) for r in carriers)

# ── Constraints ────────────────────────────────────────

# P2-C1: final profit >= initial profit  (individual rationality)
for r in carriers:
    prob2 += S1(r) + S2(r) + S3(r) >= S0[r], f"MinProfit_C{r}"

# P2-C2:total demand
prob2 += pulp.lpSum(
    x[(r, v, n)]
    for r in carriers for v in vehicles_of_carrier[r] for n in trips
) + 2 * pulp.lpSum(
    y[(r, v, n, k)]
    for r in carriers for v in vehicles_of_carrier[r]
    for n in trips for k in trips if k != n
) == len(trips), "TotalDemand"

# P2-C3: each trip covered exactly once
for n in trips:
    prob2 += pulp.lpSum(
        x[(r, v, n)]
        for r in carriers for v in vehicles_of_carrier[r]
    ) + pulp.lpSum(
        y[(r, v, n, k)] + y[(r, v, k, n)]
        for r in carriers for v in vehicles_of_carrier[r]
        for k in trips if k != n
    ) == 1, f"TripCovered_{n}"

# P2-C4: trips assigned to carrier r <= fleet size
for r in carriers:
    prob2 += pulp.lpSum(
        x[(r, v, n)] for v in vehicles_of_carrier[r] for n in trips
    ) + pulp.lpSum(
        y[(r, v, n, k)]
        for v in vehicles_of_carrier[r]
        for n in trips for k in trips if k != n
    ) <= len(vehicles_of_carrier[r]), f"FleetSize_C{r}"

# P2-C5: each vehicle executes at most one trip (or pair)
for r in carriers:
    for v in vehicles_of_carrier[r]:
        prob2 += pulp.lpSum(x[(r, v, n)] for n in trips) + pulp.lpSum(
            y[(r, v, n, k)] for n in trips for k in trips if k != n
        ) <= 1, f"OneTrip_C{r}_V{v}"

# P2-C6: battery range — single trip
for r in carriers:
    for v in vehicles_of_carrier[r]:
        if is_electric[v]:
            for n in trips:
                prob2 += (x[(r, v, n)] * TOA_single(v, n)
                          <= T_avail[v] * mu), f"Bat_single_C{r}_V{v}_N{n}"

# P2-C7: battery range — combined trip
for r in carriers:
    for v in vehicles_of_carrier[r]:
        if is_electric[v]:
            for n in trips:
                for k in trips:
                    if k != n:
                        prob2 += (y[(r, v, n, k)] * TOA_combined(v, n, k)
                                  <= T_avail[v] * mu), f"Bat_comb_C{r}_V{v}_N{n}_K{k}"

# P2-C8: combinability
for r in carriers:
    for v in vehicles_of_carrier[r]:
        for n in trips:
            for k in trips:
                if k != n:
                    prob2 += y[(r, v, n, k)] <= O[(n, k)], f"Combine_C{r}_V{v}_N{n}_K{k}"

# ── Solve ─────────────────────────────────────────
prob2.solve(pulp.PULP_CBC_CMD(msg=0))

print(f"\nStatus: {pulp.LpStatus[prob2.status]}")
print(f"Total Profit: {pulp.value(prob2.objective):.2f} €")
print()

for r in carriers:
    sr = pulp.value(S1(r) + S2(r) + S3(r))
    print(f"Carrier {r}: Final Profit = {sr:.2f} €  (Initial = {S0[r]:.2f} €)")

print("\nTrip Assignments:")
for r in carriers:
    for v in vehicles_of_carrier[r]:
        for n in trips:
            if pulp.value(x[(r, v, n)]) and pulp.value(x[(r, v, n)]) > 0.5:
                vtype = "E" if is_electric[v] else "D"
                print(f"  Single: Trip {n} → Carrier {r}, Vehicle {v} ({vtype})")
        for n in trips:
            for k in trips:
                if k != n:
                    if pulp.value(y[(r, v, n, k)]) and pulp.value(y[(r, v, n, k)]) > 0.5:
                        vtype = "E" if is_electric[v] else "D"
                        print(f"  Combined: Trips ({n},{k}) → Carrier {r}, Vehicle {v} ({vtype})")


Status: Optimal
Total Profit: 289.04 €

Carrier 0: Final Profit = 179.04 €  (Initial = 69.50 €)
Carrier 1: Final Profit = 110.00 €  (Initial = 106.40 €)

Trip Assignments:
  Combined: Trips (1,2) → Carrier 0, Vehicle 1 (D)
  Single: Trip 0 → Carrier 0, Vehicle 2 (E)
  Single: Trip 4 → Carrier 1, Vehicle 3 (E)
  Single: Trip 3 → Carrier 1, Vehicle 4 (E)


All the results

In [ ]:

for r in carriers:
  print(f"Carrier {r}: S0 = {S0[r]:.2f} €  (status: {pulp.LpStatus[prob1.status]})")


print(f"\nStatus: {pulp.LpStatus[prob2.status]}")
print(f"Total Profit: {pulp.value(prob2.objective):.2f} €")
print()

for r in carriers:
    sr = pulp.value(S1(r) + S2(r) + S3(r))
    print(f"Carrier {r}: Final Profit = {sr:.2f} €  (Initial = {S0[r]:.2f} €)")

print("\nTrip Assignments:")
for r in carriers:
    for v in vehicles_of_carrier[r]:
        for n in trips:
            if pulp.value(x[(r, v, n)]) and pulp.value(x[(r, v, n)]) > 0.5:
                vtype = "E" if is_electric[v] else "D"
                print(f"  Single: Trip {n} → Carrier {r}, Vehicle {v} ({vtype})")
        for n in trips:
            for k in trips:
                if k != n:
                    if pulp.value(y[(r, v, n, k)]) and pulp.value(y[(r, v, n, k)]) > 0.5:
                        vtype = "E" if is_electric[v] else "D"
                        print(f"  Combined: Trips ({n},{k}) → Carrier {r}, Vehicle {v} ({vtype})")

Carrier 0: S0 = 69.50 €  (status: Optimal)
Carrier 1: S0 = 106.40 €  (status: Optimal)

Status: Optimal
Total Profit: 289.04 €

Carrier 0: Final Profit = 179.04 €  (Initial = 69.50 €)
Carrier 1: Final Profit = 110.00 €  (Initial = 106.40 €)

Trip Assignments:
  Combined: Trips (1,2) → Carrier 0, Vehicle 1 (D)
  Single: Trip 0 → Carrier 0, Vehicle 2 (E)
  Single: Trip 4 → Carrier 1, Vehicle 3 (E)
  Single: Trip 3 → Carrier 1, Vehicle 4 (E)
